In [ ]:
!pip -q install pypdf
import re
import pandas as pd
from pypdf import PdfReader
from google.colab import files

up = files.upload()                      # choose the registry PDF, e.g. Registar_lijekova2026.pdf
pdf = PdfReader(list(up.keys())[0])

def parse_index_line(line):
    m = re.match(r"^(.*?)((?:,\s*\d+)+)$", line.strip())
    if not m: return None
    body = m.group(1).strip()
    if not body or body != body.lower():      # uppercase lines are brand names
        return None
    return [p.strip() for p in body.split(",")]

rows, last_page = [], None
for i in range(9, len(pdf.pages)):            # index starts around page 10 (0-based 9)
    page_rows = []
    for line in (pdf.pages[i].extract_text() or "").splitlines():
        parts = parse_index_line(line)
        if parts:
            page_rows.append({"entry": ", ".join(parts), "n_ingredients": len(parts)})
    if not page_rows and i > 15:              # first empty page after the start = end of the index
        break
    rows += page_rows
    last_page = i

built = pd.DataFrame(rows).drop_duplicates()
built.to_csv("/content/registry_substances.csv", index=False)
print("Index pages (0-based): 9 to", last_page)
print(len(built), "entries;", (built.n_ingredients == 1).sum(), "single-substance")
print("Alphabetical tail (should end in z-names):")
print(built.sort_values("entry").tail(5).to_string(index=False))

Saving Registar_lijekova2026.pdf to Registar_lijekova2026 (1).pdf
Index pages (0-based): 9 to 80
909 entries; 682 single-substance
Alphabetical tail (should end in z-names):
               entry  n_ingredients
          ziprazidon              1
          zofenopril              1
zoledronska kiselina              1
        zolmitriptan              1
            zolpidem              1


In [ ]:
from pathlib import Path
import re
import pandas as pd

# ---- CONFIG (Colab defaults) ----
REGISTRY_PATH = Path("/content/registry_substances.csv")
NAME_COL = "entry"
SUBSTANCE_COL = "entry"
DDINTER_DIR = Path("/content/ddinter")   # re-upload the 8 DDInter CSVs here if the session was reset

## 1. Load the registry

In [ ]:
def load_table(path):
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    for enc in ("utf-8", "cp1250", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, sep=None, engine="python")
        except Exception as e:
            last = e
    raise last

reg = load_table(REGISTRY_PATH)
print("Rows:", len(reg), " Columns:", len(reg.columns))
print(list(reg.columns))
reg.head(10)

Rows: 909  Columns: 2
['entry', 'n_ingredients']


,entry,n_ingredients
0,"2, 4-dihlorbenzil alkohol, amilmetakrezol",3
1,"2, 4-dihlorbenzil alkohol, amilmetakrezol, ask...",4
2,"2, 4-dihlorbenzil alkohol, amilmetakrezol, lev...",4
3,"2, 4-dihlorbenzil alkohol, amilmetakrezol, lid...",4
4,"abakavir, dolutegravir, lamivudin",3
5,"abakavir, lamivudin",2
6,abemaciklib,1
7,abirateron,1
8,abrocitinib,1
9,aceklofenak,1


In [ ]:
print("Missing values per column:")
print(reg.isna().sum())
print("\nDuplicate rows:", reg.duplicated().sum())
assert NAME_COL in reg.columns, f"Set NAME_COL to one of: {list(reg.columns)}"

Missing values per column:
entry            0
n_ingredients    0
dtype: int64

Duplicate rows: 0


## 2. What do the name strings look like?


In [ ]:
names = reg[NAME_COL].dropna().astype(str)
print(names.sample(min(15, len(names)), random_state=0).to_string(index=False))

feat = pd.DataFrame({
    "has_digits":      names.str.contains(r"\d"),
    "has_strength":    names.str.contains(r"\b\d+([.,]\d+)?\s*(mg|g|mcg|µg|ml|iu|ij)\b", case=False),
    "has_form_words":  names.str.contains(r"tablet|kapsul|sirup|otopin|mast|krema|gel|supozit|prašak|prasak", case=False),
    "has_combination": names.str.contains(r"\+|/"),
    "has_diacritics":  names.str.contains(r"[čćžšđČĆŽŠĐ]"),
    "has_latin_ending":names.str.contains(r"\b\w+(um|i hydrochloridum|i natricus)\b", case=False),
})
print("\nShare of strings with each feature:")
print(feat.mean().round(3).to_string())

                                        dimetinden
                                       linagliptin
                                          joversol
                                        olopatadin
                                          bilastin
                               donepezil, memantin
aluminijum-hidroksid, magnezijum-karbonat, magn...
                                         mesalazin
                                   natrijum-hlorid
                          acetilsalicilna kiselina
                                      fremanezumab
                                        flukonazol
                                        flutikazon
                                        propafenon
                                        pirfenidon

Share of strings with each feature:
has_digits          0.010
has_strength        0.000
has_form_words      0.001
has_combination     0.000
has_diacritics      0.026
has_latin_ending    0.051


/tmp/ipykernel_2021/49039768.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_strength":    names.str.contains(r"\b\d+([.,]\d+)?\s*(mg|g|mcg|µg|ml|iu|ij)\b", case=False),
/tmp/ipykernel_2021/49039768.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "has_latin_ending":names.str.contains(r"\b\w+(um|i hydrochloridum|i natricus)\b", case=False),


## 3. How many match DDInter with NO normalization?


In [ ]:
files = sorted(DDINTER_DIR.glob("*.csv"))
assert files, f"No DDInter CSVs in {DDINTER_DIR.resolve()}"
dd = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def find_col(df, pats):
    for c in df.columns:
        if any(re.search(p, c, re.I) for p in pats): return c
col_a = find_col(dd, [r"^drug_?a$"]); col_b = find_col(dd, [r"^drug_?b$"])
norm = lambda s: re.sub(r"\s+", " ", str(s).strip().lower())
ddi_names = set(dd[col_a].dropna().map(norm)) | set(dd[col_b].dropna().map(norm))
print("DDInter unique medicine names:", len(ddi_names))

raw_match = names.map(norm).isin(ddi_names)
print(f"\nProduct-name strings that exactly match a DDInter name: {raw_match.sum()} of {len(names)} ({raw_match.mean():.1%})")

if SUBSTANCE_COL and SUBSTANCE_COL in reg.columns:
    subs = reg[SUBSTANCE_COL].dropna().astype(str)
    sub_match = subs.map(norm).isin(ddi_names)
    print(f"Active-substance strings that exactly match a DDInter name: {sub_match.sum()} of {len(subs)} ({sub_match.mean():.1%})")
    print("\nExamples that do NOT match (spelling, Latin, salts, combinations):")
    print(subs[~sub_match].sample(min(15, (~sub_match).sum()), random_state=0).to_string(index=False))
else:
    print("\nNo active-substance column set: the registry may not list ingredients. Note this in the data card.")

DDInter unique medicine names: 1939

Product-name strings that exactly match a DDInter name: 165 of 909 (18.2%)
Active-substance strings that exactly match a DDInter name: 165 of 909 (18.2%)

Examples that do NOT match (spelling, Latin, salts, combinations):
                                      tenekteplaza
mentol, metil-salicilat, terpentinsko ulje, ulj...
                          acetilsalicilna kiselina
                                         aksitinib
                                        cefotaksim
                                        apalutamid
                          glukoza, natrijum-hlorid
                                         ampicilin
                                        vericigvat
                                   agalzidaza beta
                                       eritromicin
                                      hloropiramin
                            ibuprofen, paracetamol
                                     ciklofosfamid
alfa pinen, anetol, beta pi

In [ ]:
single = built[built.n_ingredients == 1]["entry"].map(norm)
print("Single-substance entries:", len(single))
print(f"Of those, exact match with DDInter: {single.isin(ddi_names).sum()} ({single.isin(ddi_names).mean():.1%})")
print("Combination entries:", (built.n_ingredients > 1).sum())

Single-substance entries: 682
Of those, exact match with DDInter: 165 (24.2%)
Combination entries: 227


## 4. Summary for the handout

In [ ]:
print("=== Registry numbers ===")
print("Rows / columns:           ", len(reg), "/", len(reg.columns))
print("Has active-substance col: ", bool(SUBSTANCE_COL and SUBSTANCE_COL in reg.columns))
print("Raw name match vs DDInter:", f"{raw_match.mean():.1%}")
print("Strings with strength/form/combination:",
      f"{feat['has_strength'].mean():.1%} / {feat['has_form_words'].mean():.1%} / {feat['has_combination'].mean():.1%}")

=== Registry numbers ===
Rows / columns:            909 / 2
Has active-substance col:  True
Raw name match vs DDInter: 18.2%
Strings with strength/form/combination: 0.0% / 0.1% / 0.0%
